# From Red Flags to Detection Rules
## An LLM-driven Pipeline for Real-Time GOOSE Intrusion Detection and Prevention

> **Autores:** Lucas A. Martins¹*, Silvio E. Quincozes¹²  
> ¹ Universidade Federal de Uberlândia (UFU) – Uberlândia, Brasil  
> ² Universidade Federal do Pampa (UNIPAMPA) – Alegrete, Brasil  
> `{lucas.martins, sequincozes}@ufu.br`

---

### Resumo

Sistemas de Detecção de Intrusão (IDS) baseados em especificação são amplamente utilizados em subestações IEC 61850, mas dependem de regras criadas manualmente por especialistas. Este notebook apresenta um **pipeline orientado por LLM** que automatiza a geração de regras de detecção para detecção e prevenção de intrusões GOOSE em tempo real.

A abordagem utiliza amostras de comunicação rotuladas para identificar *red flags*, que são transformadas em regras de detecção de intrusão executáveis. A prova de conceito usa o **dataset ERENO** e demonstra que as regras geradas detectam comportamentos anômalos com baixo overhead operacional.

## Índice

1. [Introdução](#1-introdução)
2. [IDS Baseado em Especificação para GOOSE](#2-ids-baseado-em-especificação-para-goose)
3. [Arquitetura Proposta](#3-arquitetura-proposta)
4. [Instalação e Configuração do Ambiente](#4-instalação-e-configuração-do-ambiente)
5. [Ingestão de Dados (ERENO)](#5-ingestão-de-dados-ereno)
6. [Extração de Red Flags via LLM](#6-extração-de-red-flags-via-llm)
7. [Geração de Regras de Detecção](#7-geração-de-regras-de-detecção)
8. [Simulação do Switch Programável](#8-simulação-do-switch-programável)
9. [Avaliação: Capacidade de Detecção](#9-avaliação-capacidade-de-detecção)
10. [Avaliação: Viabilidade em Tempo Real](#10-avaliação-viabilidade-em-tempo-real)
11. [Considerações Finais](#11-considerações-finais)
12. [Referências](#12-referências)

---
## 1. Introdução

Subestações digitais baseadas no padrão **IEC–61850** enfrentam desafios crescentes de cibersegurança, incluindo ataques de:

- **Denial-of-Service (DoS)**
- **Injeção de mensagens (Message Injection)**
- **Mascaramento (Masquerade attacks)**

IDS baseados em especificação são atraentes nesse contexto por seu **baixo overhead computacional** e **interpretabilidade**. No entanto, dependem de regras escritas manualmente por especialistas — processo custoso, difícil de escalar e pouco adaptável.

**Motivação principal:** Automatizar a geração dessas regras usando LLMs a partir de amostras rotuladas do dataset ERENO–IEC–61850.

---
## 2. IDS Baseado em Especificação para GOOSE

O protocolo **GOOSE (Generic Object Oriented Substation Event)**, definido pelo padrão IEC–61850-8-1, suporta operações de proteção e controle em tempo crítico via modelo publisher/subscriber sobre Ethernet.

### Campos relevantes de um frame GOOSE

| Categoria | Campos |
|-----------|--------|
| Estruturais | `dst_mac`, `TPID`, `ethType`, `gooseAppid`, `timeAllowedToLive` |
| Consistência | `gocbRef`, `datSet`, `goID`, `t`, `stNum`, `sqNum` |
| Dinâmica | frequência de mensagens, bytes/s, pacotes/s |

> ⚠️ O GOOSE não foi projetado com mecanismos nativos de segurança robustos, tornando-o vulnerável a ataques de injeção, replay e negação de serviço.

---
## 3. Arquitetura Proposta

O pipeline é composto por **quatro estágios principais**:

```
┌─────────────────┐    ┌──────────────────┐    ┌───────────────────┐    ┌──────────────────┐
│  Dataset GOOSE  │───▶│ Red Flag Extract.│───▶│  Rule Generation  │───▶│ Switch Simulation│
│  Rotulado       │    │ (LLM-based)      │    │  (Python rules)   │    │  (Real-time)     │
│  (ERENO)        │    │                  │    │                   │    │                  │
└─────────────────┘    └──────────────────┘    └───────────────────┘    └──────────────────┘
```

| Estágio | Responsabilidade |
|---------|------------------|
| **1. Source Ingestion** | Carrega dataset, organiza features, prepara prompts estruturados |
| **2. Red Flag Extraction** | LLM identifica padrões suspeitos e inconsistências comportamentais |
| **3. Rule Generation** | Traduz red flags em regras Python executáveis |
| **4. Simulated Deployment** | Executa regras sobre tráfego GOOSE em tempo real simulado |

---
## 4. Instalação e Configuração do Ambiente

In [ ]:
# Clone do repositório principal
!git clone https://github.com/sequincozes/CounselorNode.git
%cd CounselorNode

In [ ]:
# Instalação das dependências (recomenda-se ambiente virtual)
!pip install -r requirements.txt

In [ ]:
# Imports principais utilizados ao longo do pipeline
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Para integração com LLM (ex: OpenAI, Anthropic, ou modelo local)
# import openai  # descomente conforme o provedor utilizado

print("Ambiente configurado com sucesso.")

---
## 5. Ingestão de Dados (ERENO)

O dataset **ERENO–IEC–61850** fornece amostras rotuladas de tráfego GOOSE em condições normais e sob ataque.

Cada amostra contém features de três tipos:
- **Protocol-level:** campos do protocolo GOOSE
- **Temporal:** diferenças de timestamp entre mensagens
- **Derived:** métricas calculadas como `stDiff`, `sqDiff`, `timestampDiff`

In [ ]:
# Carregamento do dataset ERENO
# Substitua pelo caminho real do arquivo CSV/PCAP após obter o dataset
DATASET_PATH = Path("data/ereno_goose.csv")

# Exemplo de estrutura esperada:
example_columns = [
    "stNum", "sqNum", "timestamp", "stDiff", "sqDiff", "timestampDiff",
    "gocbRef", "datSet", "goID", "timeAllowedToLive",
    "src_mac", "dst_mac", "gooseAppid", "bytes_per_sec", "pkts_per_sec",
    "label"  # 0 = normal, 1 = ataque
]

print("Colunas esperadas no dataset:")
for col in example_columns:
    print(f"  - {col}")

In [ ]:
# Carrega o dataset se disponível
if DATASET_PATH.exists():
    df = pd.read_csv(DATASET_PATH)
    print(f"Dataset carregado: {df.shape[0]} amostras, {df.shape[1]} features")
    print("\nDistribuição das classes:")
    print(df["label"].value_counts())
    display(df.head())
else:
    print("[AVISO] Dataset não encontrado. Ajuste DATASET_PATH para o caminho correto.")
    print("Referência: Quincozes et al. (2022) — ERENO dataset")

In [ ]:
def prepare_llm_prompt(df_sample: pd.DataFrame, n_normal: int = 5, n_attack: int = 5) -> str:
    """
    Prepara um prompt estruturado para o LLM a partir de amostras rotuladas.
    
    Args:
        df_sample: DataFrame com amostras do dataset ERENO
        n_normal: número de amostras normais a incluir no prompt
        n_attack: número de amostras de ataque a incluir no prompt
    
    Returns:
        String com o prompt formatado para extração de red flags
    """
    normal_samples = df_sample[df_sample["label"] == 0].head(n_normal)
    attack_samples = df_sample[df_sample["label"] == 1].head(n_attack)

    prompt = """Você é um especialista em segurança de subestações digitais IEC 61850.
Analise as amostras de tráfego GOOSE abaixo e identifique RED FLAGS —
condições ou padrões que distinguem tráfego malicioso do benigno.

=== AMOSTRAS NORMAIS ===
"""
    prompt += normal_samples.to_string(index=False)
    prompt += "\n\n=== AMOSTRAS DE ATAQUE ===\n"
    prompt += attack_samples.to_string(index=False)
    prompt += """

Liste as red flags identificadas no formato:
- RED FLAG: <descrição da condição anômala>
  CAMPO(S): <campo(s) do protocolo envolvido(s)>
  RAZÃO: <por que isso indica um ataque>
"""
    return prompt


# Exemplo de saída do prompt (para visualização)
print("Estrutura do prompt para o LLM:")
print("-" * 60)
print("[AMOSTRAS NORMAIS] → [AMOSTRAS DE ATAQUE] → Solicita red flags")

---
## 6. Extração de Red Flags via LLM

O LLM recebe descrições estruturadas das amostras rotuladas e produz **red flags** em formato semi-estruturado.

Exemplos de red flags esperadas:
- Transições anômalas em `stNum` ou `sqNum`
- Comportamento temporal inconsistente (`timestampDiff` fora do padrão)
- Combinações suspeitas de campos do protocolo
- Variações abruptas em `stDiff`, `sqDiff`

In [ ]:
def call_llm_for_red_flags(prompt: str, model: str = "gpt-4") -> str:
    """
    Envia o prompt ao LLM e retorna as red flags extraídas.
    
    Adapte esta função ao provedor de LLM utilizado
    (OpenAI, Anthropic Claude, modelo local via Ollama, etc.)
    
    Args:
        prompt: prompt estruturado com amostras rotuladas
        model: identificador do modelo LLM
    
    Returns:
        String com as red flags geradas pelo LLM
    """
    # --- OpenAI ---
    # import openai
    # response = openai.chat.completions.create(
    #     model=model,
    #     messages=[{"role": "user", "content": prompt}]
    # )
    # return response.choices[0].message.content

    # --- Anthropic Claude ---
    # import anthropic
    # client = anthropic.Anthropic()
    # message = client.messages.create(
    #     model="claude-opus-4-6",
    #     max_tokens=1024,
    #     messages=[{"role": "user", "content": prompt}]
    # )
    # return message.content[0].text

    raise NotImplementedError("Configure o provedor de LLM desejado acima.")


# Exemplo de red flags (saída simulada para referência)
EXAMPLE_RED_FLAGS = """
- RED FLAG: stNum não incrementa de forma monotônica
  CAMPO(S): stNum, stDiff
  RAZÃO: Em operação normal, stNum aumenta a cada mudança de estado; decremento indica replay ou injeção.

- RED FLAG: sqNum reinicia inesperadamente sem mudança em stNum
  CAMPO(S): sqNum, sqDiff, stNum
  RAZÃO: sqNum deve reiniciar apenas quando stNum incrementa; reinício isolado sugere frame forjado.

- RED FLAG: timestampDiff negativo ou muito elevado
  CAMPO(S): timestampDiff, timestamp
  RAZÃO: Mensagens GOOSE têm timing previsível; desvios extremos indicam replay ou flood.

- RED FLAG: pacotes por segundo acima do limiar esperado
  CAMPO(S): pkts_per_sec
  RAZÃO: Volume anormal de tráfego pode indicar ataque de DoS ou flood.
"""

print("Exemplo de red flags extraídas pelo LLM:")
print(EXAMPLE_RED_FLAGS)

---
## 7. Geração de Regras de Detecção

As red flags extraídas são convertidas em **regras Python executáveis**. Cada regra é uma função condicional que recebe um pacote GOOSE (como dicionário) e retorna `True` se a condição de ataque for satisfeita.

As regras seguem o mesmo espírito das regras IDS baseadas em especificação tradicionais, operando sobre:
- Campos do protocolo
- Consistência entre mensagens
- Métricas de nível de tráfego

In [ ]:
# =============================================================
# Regras de detecção geradas pelo pipeline LLM
# Cada função recebe um dicionário representando um frame GOOSE
# e retorna True se o tráfego for considerado suspeito
# =============================================================

def rule_stnum_non_monotonic(pkt: dict) -> bool:
    """Red flag: stNum não incrementa de forma monotônica."""
    return pkt.get("stDiff", 0) < 0


def rule_sqnum_unexpected_reset(pkt: dict) -> bool:
    """Red flag: sqNum reinicia sem mudança correspondente em stNum."""
    return pkt.get("sqDiff", 0) < 0 and pkt.get("stDiff", 0) == 0


def rule_timestamp_anomaly(pkt: dict, max_diff_ms: float = 5000.0) -> bool:
    """Red flag: diferença de timestamp negativa ou excessivamente grande."""
    ts_diff = pkt.get("timestampDiff", 0)
    return ts_diff < 0 or ts_diff > max_diff_ms


def rule_packet_flood(pkt: dict, threshold_pps: float = 1000.0) -> bool:
    """Red flag: taxa de pacotes por segundo acima do limiar esperado."""
    return pkt.get("pkts_per_sec", 0) > threshold_pps


def rule_goose_field_inconsistency(pkt: dict) -> bool:
    """Red flag: inconsistência entre gocbRef e datSet esperados."""
    expected_gocbRef = pkt.get("expected_gocbRef", None)
    if expected_gocbRef is None:
        return False
    return pkt.get("gocbRef") != expected_gocbRef


# Lista de todas as regras ativas
DETECTION_RULES = [
    rule_stnum_non_monotonic,
    rule_sqnum_unexpected_reset,
    rule_timestamp_anomaly,
    rule_packet_flood,
    rule_goose_field_inconsistency,
]

print(f"{len(DETECTION_RULES)} regras de detecção carregadas.")
for r in DETECTION_RULES:
    print(f"  ✓ {r.__name__}: {r.__doc__}")

---
## 8. Simulação do Switch Programável

O componente de switch simula um ambiente de processamento de pacotes em tempo real. Para cada amostra GOOSE processada, o simulador:

1. Aplica o conjunto de regras gerado
2. Determina se o tráfego é benigno ou suspeito
3. Registra a decisão nos logs de execução

> As regras são condições Python leves sobre atributos de pacotes — preservando o baixo overhead esperado em subestações digitais.

In [ ]:
from typing import List, Callable


class GOOSESwitchSimulator:
    """
    Simula um switch programável para detecção e prevenção de intrusões GOOSE.
    Aplica o conjunto de regras gerado pelo pipeline LLM sobre amostras de tráfego.
    """

    def __init__(self, rules: List[Callable]):
        self.rules = rules
        self.execution_log = []

    def process_packet(self, pkt: dict) -> dict:
        """
        Processa um pacote GOOSE e retorna a decisão de detecção/prevenção.
        
        Returns:
            dict com campos: packet_id, triggered_rules, decision (ALLOW/BLOCK)
        """
        triggered = []
        for rule in self.rules:
            try:
                if rule(pkt):
                    triggered.append(rule.__name__)
            except Exception as e:
                triggered.append(f"{rule.__name__}:ERROR({e})")

        decision = "BLOCK" if triggered else "ALLOW"
        result = {
            "packet_id": pkt.get("id", "unknown"),
            "triggered_rules": triggered,
            "decision": decision,
            "true_label": pkt.get("label", None),
        }
        self.execution_log.append(result)
        return result

    def run(self, packets: List[dict]) -> pd.DataFrame:
        """Processa uma lista de pacotes e retorna o log de execução como DataFrame."""
        self.execution_log = []
        for pkt in packets:
            self.process_packet(pkt)
        return pd.DataFrame(self.execution_log)


# Instancia o simulador com as regras geradas
simulator = GOOSESwitchSimulator(rules=DETECTION_RULES)
print("Simulador inicializado com", len(DETECTION_RULES), "regras.")

In [ ]:
# Teste com pacotes sintéticos representativos
test_packets = [
    # Pacote normal
    {"id": 1, "stDiff": 1, "sqDiff": 1, "timestampDiff": 4.0,
     "pkts_per_sec": 50.0, "label": 0},
    # Ataque: stNum regressivo (replay)
    {"id": 2, "stDiff": -3, "sqDiff": 0, "timestampDiff": 4.0,
     "pkts_per_sec": 50.0, "label": 1},
    # Ataque: flood de pacotes
    {"id": 3, "stDiff": 0, "sqDiff": 1, "timestampDiff": 0.5,
     "pkts_per_sec": 5000.0, "label": 1},
    # Ataque: sqNum reseta sem mudança de stNum
    {"id": 4, "stDiff": 0, "sqDiff": -10, "timestampDiff": 4.0,
     "pkts_per_sec": 50.0, "label": 1},
    # Pacote normal — limite de frequência
    {"id": 5, "stDiff": 0, "sqDiff": 1, "timestampDiff": 3.8,
     "pkts_per_sec": 999.9, "label": 0},
]

results_df = simulator.run(test_packets)
print("Resultados da simulação:")
display(results_df)

---
## 9. Avaliação: Capacidade de Detecção

Avalia as regras geradas comparando as decisões do simulador com os rótulos verdadeiros do dataset.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


def evaluate_detection(results: pd.DataFrame) -> None:
    """
    Calcula e exibe métricas de detecção: precisão, recall, F1 e matriz de confusão.
    """
    if results["true_label"].isna().all():
        print("Rótulos verdadeiros não disponíveis. Forneça o dataset rotulado.")
        return

    y_true = results["true_label"].astype(int)
    y_pred = (results["decision"] == "BLOCK").astype(int)

    print("=== Relatório de Classificação ===")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Ataque"]))

    print("=== Matriz de Confusão ===")
    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Real: Normal", "Real: Ataque"],
        columns=["Pred: Normal", "Pred: Ataque"]
    )
    display(cm_df)


evaluate_detection(results_df)

---
## 10. Avaliação: Viabilidade em Tempo Real

Mede a latência de detecção por pacote para verificar se o overhead é adequado para subestações digitais.

In [ ]:
import time


def benchmark_latency(simulator: GOOSESwitchSimulator,
                      packets: List[dict],
                      n_runs: int = 1000) -> dict:
    """
    Mede a latência média de processamento por pacote.
    
    Args:
        simulator: instância do GOOSESwitchSimulator
        packets: lista de pacotes de teste
        n_runs: número de repetições para estimativa estável
    
    Returns:
        dict com latência média, mínima e máxima em microsegundos
    """
    latencies = []
    for _ in range(n_runs):
        for pkt in packets:
            t0 = time.perf_counter()
            simulator.process_packet(pkt)
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1e6)  # microsegundos

    return {
        "mean_us": np.mean(latencies),
        "min_us": np.min(latencies),
        "max_us": np.max(latencies),
        "p99_us": np.percentile(latencies, 99),
    }


latency_stats = benchmark_latency(simulator, test_packets, n_runs=500)

print("=== Latência de Processamento por Pacote ===")
for k, v in latency_stats.items():
    print(f"  {k}: {v:.2f} µs")

---
## 11. Considerações Finais

Este trabalho apresentou um pipeline orientado por LLM que automatiza a geração de regras de detecção para IDS baseados em especificação em subestações IEC 61850.

### Contribuições principais

- **Pipeline plug-and-play** end-to-end: do dataset rotulado às regras executáveis
- **Redução da dependência de especialistas** para criação manual de regras
- **Baixo overhead operacional**: regras Python leves adequadas a ambientes em tempo real
- **Reprodutibilidade**: parâmetros configuráveis e artefatos rastreáveis

### Limitações e trabalhos futuros

- Validação em hardware real de subestação (não apenas simulado)
- Avaliação com mais classes de ataque do dataset ERENO
- Comparação com abordagens de ML supervisionado
- Adaptação automática a novos padrões de ataque via atualização incremental

> **Repositório:** https://github.com/sequincozes/CounselorNode

---
## 12. Referências

- **Commission, I. E.** (2003). Communication networks and systems in substations - Part 8-1: SCSM. IET.

- **Hong, J. and Liu, C.** (2019). Intelligent electronic devices with collaborative intrusion detection systems. *IEEE Transactions on Smart Grid*, 10(1):271–281.

- **Hong, J., Liu, C., and Govindarasu, M.** (2014a). Detection of Cyber Intrusions Using Network-Based Multicast Messages for Substation Automation. *ISGT*, IEEE.

- **Hong, J., Liu, C., and Govindarasu, M.** (2014b). Integrated anomaly detection for cyber security of the substations. *IEEE Transactions on Smart Grid*, 5(4):1643–1653.

- **Kwon, Y. et al.** (2015). A behavior-based intrusion detection technique for smart grid infrastructure. *IEEE Eindhoven PowerTech*.

- **Malik, H., Alotaibi, M. A., and Almutairi, A.** (2022). Cyberattacks identification in IEC 61850 based substation using proximal SVM. *Journal of Intelligent & Fuzzy Systems*, 42(2):1213–1222.

- **Quincozes, S. E. et al.** (2021). A survey on intrusion detection and prevention systems in digital substations. *Computer Networks*, 184:107679.

- **Quincozes, S. E. et al.** (2022). ERENO: An Extensible Tool for Generating Realistic IEC–61850 Intrusion Detection Datasets. PhD thesis, UFF.

- **Yang, Y. et al.** (2016a). Intrusion detection system for IEC 61850 based smart substations. *IEEE PESGM*.

- **Yang, Y. et al.** (2016b). Multidimensional intrusion detection system for IEC 61850-based SCADA networks. *IEEE Transactions on Power Delivery*, 32(2):1068–1078.